# 05 · What is *quantization*?

**Recap of the LoRA story (notebooks 01–04):** a layer is `output = W @ x`; to
fine-tune cheaply we **freeze** the giant pretrained `W` and train only a small
low-rank change `B·A`.

**QLoRA changes exactly one thing:** *how the frozen `W` is stored.* Before we can
see that change, we need one new idea — **quantization** — and that's all this
notebook is. Tiny numbers, as always. Let's go!

In [ ]:
import numpy as np

## Step 1 — Numbers cost bits, and bits limit how many values you can have

A computer stores a number using **bits**. With `n` bits you can represent
`2^n` distinct values. More bits → more possible values → more precision, but
more memory used.

In [ ]:
for bits in (2, 4, 8, 16):
    print(f"{bits:>2} bits  ->  {2**bits:>5} distinct values")

So **4 bits gives you only 16 different values.** That's the entire vocabulary a
4-bit number gets to choose from. Hold that — it's the whole tension in QLoRA.

## Step 2 — Quantization = snap a number to the nearest *allowed* value

Suppose the only values we're allowed to store are these five:

$$\{\,0,\;\; 0.25,\;\; 0.5,\;\; 0.75,\;\; 1.0\,\}$$

To store any other number, we **round it to the nearest allowed one**. That
rounding *is* quantization. The little gap left behind is the **quantization
error**. By hand: to store `0.7`, the nearest allowed value is `0.75`, so we keep
`0.75` and the error is `0.05`.

In [ ]:
grid = np.array([0, 0.25, 0.5, 0.75, 1.0])    # the only values we may store

def quantize(v):
    return grid[np.argmin(np.abs(grid - v))]   # nearest allowed value

for v in [0.7, 0.1, 0.62]:
    q = quantize(v)
    print(f"store {v}  ->  nearest allowed {q}   (error {abs(v - q):.2f})")

## Step 3 — More allowed values → smaller error

Now the key trade-off, shown directly. We take 10,000 pretend "weights" and store
them using `L` evenly-spaced allowed values. Watch the average error shrink as we
allow more values:

In [ ]:
rng = np.random.default_rng(0)
w = rng.uniform(-1, 1, size=10_000)    # pretend these are a layer's weights

def quantize_to_levels(w, L):
    grid = np.linspace(w.min(), w.max(), L)          # L evenly-spaced values
    idx = np.abs(w[:, None] - grid[None, :]).argmin(axis=1)
    return grid[idx]

for L in (4, 16, 256):
    wq = quantize_to_levels(w, L)
    print(f"{L:>3} allowed values  ->  mean |error| = {np.abs(w - wq).mean():.4f}")

Fewer values = coarser rounding = bigger error. **16 values (4 bits) is noticeably
rougher than 256 (8 bits).** That's the price of going to 4 bits.

## Step 4 — Why we'd pay that price: memory

Why tolerate the extra error at all? Because storing weights in 4 bits instead of
16 makes the model **4× smaller** — and that's the difference between fitting on a
small GPU or not.

In [ ]:
params = 1.5e9     # a 1.5-billion-parameter model
for bits in (16, 4):
    gb = params * bits / 8 / 1e9
    print(f"{bits:>2}-bit weights  ->  {gb:.2f} GB just to hold the model")

3 GB vs 0.75 GB, for the *same* model — purely a storage choice.

## Recap

- A number is stored in **bits**; `n` bits → `2^n` possible values. **4 bits → 16 values.**
- **Quantization** = round each number to the nearest allowed value; the leftover
  gap is **quantization error**.
- **Fewer allowed values → more error**, but **far less memory** (4-bit is 4× smaller than 16-bit).

The obvious worry: 16 values is *rough*. Next notebook shows the clever fix —
**NF4**, a 4-bit grid whose 16 values are placed to match how neural-net weights
are actually distributed, so the error is much smaller than the evenly-spaced grid
we used here.

**BAM!** Next: **`06 — 4-bit and NF4`.**